# hp-adaptivity on the L-shaped domain

The curl–curl problem on the L-shape with the corner singularity $E = \nabla(r^{2/3}
\sin\tfrac{2\theta}{3})$ as the exact total field (prescribed on the boundary, driven
by the matching current). The loop estimates, marks (Dörfler), decides between $h$ and
$p$ by error prediction, refines and transfers the solution. The error should decay
exponentially in $N^{1/3}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import hpfem
from hpfem import interop

hpfem.set_log_level("warn")
K = 1.0
BOUNDARY = 9

In [ ]:
def l_shape(n):
    square = hpfem.rectangle(2 * n, 2 * n, [-1.0, -1.0], [1.0, 1.0])
    mesh = hpfem.extract(square, lambda x: not (x[0] > 0 and x[1] < 0))
    for f in mesh.boundary_facets:
        mesh.set_facet_tag(f, BOUNDARY)
    return mesh


def singular_value(x):
    nu = 2.0 / 3.0
    r = np.hypot(x[0], x[1])
    if r == 0:
        return np.zeros(2, dtype=complex)
    theta = np.arctan2(x[1], x[0])
    theta = theta + 2 * np.pi if theta < 0 else theta
    dr = nu * r ** (nu - 1) * np.sin(nu * theta)
    dt = nu * r ** (nu - 1) * np.cos(nu * theta)
    return np.array([np.cos(theta) * dr - np.sin(theta) * dt,
                     np.sin(theta) * dr + np.cos(theta) * dt], dtype=complex)


exact = hpfem.IncidentField2D(singular_value, lambda x: np.zeros(1, dtype=complex))


def setup():
    s = hpfem.ScatteringSetup2D()
    s.omega = K * hpfem.constants.c0
    s.incident = exact
    s.incident_tags = [BOUNDARY]
    s.current = lambda x: -(K**2) * singular_value(x)  # curl curl E - k^2 E = f, curl E = 0
    return s

## The loop

In [ ]:
adaptive = hpfem.AdaptiveMesh2D(l_shape(2))
orders = np.ones(adaptive.mesh.num_cells, dtype=int)
predicted = np.zeros(0)
history = []
for step in range(12):
    mesh = adaptive.mesh                      # copy of the current leaf mesh
    dofs = hpfem.NedelecDofMap2D(mesh, orders.tolist())
    problem = hpfem.Scattering2D(dofs, setup())
    solution = problem.solve()
    error = problem.error(solution, exact)
    estimate = problem.estimate(solution)
    history.append((dofs.num_dofs, np.hypot(error.l2, error.curl), estimate.total()))
    marked = hpfem.dorfler_marking(estimate.indicators, 0.5)
    decision = hpfem.hp_decide_by_prediction(estimate.indicators, predicted, marked)
    print(f"step {step:2d}: {dofs.num_dofs:6d} DoF, error {history[-1][1]:.3e}, "
          f"eta {estimate.total():.3e}, h {len(decision.h_marked):3d}, p {len(decision.p_marked):3d}, "
          f"max p {dofs.max_order}")
    hp = hpfem.hp_refine(adaptive, orders.tolist(), decision.h_marked, decision.p_marked)
    predicted = hpfem.predict_indicators(estimate.indicators, orders.tolist(), hp)
    orders = hp.orders

## Convergence and the final mesh

Exponential convergence is a straight line against $N^{1/3}$; the mesh is graded towards
the re-entrant corner with low orders there and high orders away from it.

In [ ]:
n, err, eta = np.array(history).T
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
interop.plot_convergence(n, err, ax=axes[0], exponent=1 / 3, label="error")
interop.plot_convergence(n, eta, ax=axes[0], exponent=1 / 3, label="estimate")
mesh = adaptive.mesh
tri_mesh = mesh
axes[1].tripcolor(mesh.vertices[:, 0], mesh.vertices[:, 1], mesh.cells, facecolors=orders, cmap="viridis")
axes[1].triplot(mesh.vertices[:, 0], mesh.vertices[:, 1], mesh.cells, color="k", linewidth=0.3)
axes[1].set_aspect("equal"); axes[1].set_title("polynomial order per cell")
plt.tight_layout()